### Libraries



In [ ]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

!pip install -U bitsandbytes

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, BitsAndBytesConfig, pipeline
from peft import PeftModel
from huggingface_hub import hf_hub_download, login
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import json
import time
import psutil
import os

### Login to huggingface

In [ ]:
login()

### Load models

In [ ]:
device = "cuda" # Used in colab
#device = "cpu"   # Used locally

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-sentiment-4bit-v7"
subfolder = "adapters/epoch_005"
bert_model_id = "oliverguhr/german-sentiment-bert"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


#### Fine-tuned model

In [ ]:
# 4-bit quantization config
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

# Load base model in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quant_config,
    device_map="auto",
)

# Attach LoRA adapters
lora_model = PeftModel.from_pretrained(base_model, lora_repo_id, subfolder=subfolder)
#lora_model = PeftModel.from_pretrained(base_model, lora_repo_id)

lora_model = lora_model.to(device)
lora_model.eval()

adapters/epoch_005/adapter_model.safeten(…):   0%|          | 0.00/61.3M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.058, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Lin

In [ ]:
print(lora_model.base_model_prefix)
print(lora_model)

#### Base model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(base_model_id)
model = model.to(device)
model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

#### BERT model

In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(bert_model_id)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

### Download test data



In [ ]:
def download_data(repo_id, filename):
    # Download the JSONL file from Hugging Face Hub
    json_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset"
    )

    # Load full JSONL
    with open(json_path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

    # Convert to DataFrame and clean
    df = pd.DataFrame(data)
    df = df[["review_text", "sentiment"]]
    df = df.dropna(subset=["review_text", "sentiment"])
    df["review_text"] = df["review_text"].astype(str).str.strip()
    df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
    df = df[df["sentiment"].isin(["positive", "neutral", "negative"])]


    # Sample 1000 from each class
    df_balanced = None
    df_balanced = pd.concat([
        df[df["sentiment"] == "positive"].sample(n=1000, random_state=333),
        df[df["sentiment"] == "neutral"].sample(n=1000, random_state=333),
        df[df["sentiment"] == "negative"].sample(n=1000, random_state=333)
    ], ignore_index=True).sample(frac=1, random_state=333)  # Shuffle after combining

    return df_balanced

    #return df

#df_test = download_data("eduhuemar001/dataset-sentiment-4bit-test-v7", "test_dataset.json")
df_balanced_test  = download_data("eduhuemar001/dataset-sentiment-4bit-test-v7", "test_dataset.json")
#df_balanced_train = download_data("eduhuemar001/dataset-sentiment-4bit-train-v6", "train_used_dataset.json")
df_balanced_train = download_data("eduhuemar001/dataset-sentiment-4bit-train-v7", "train_dataset.json")
#df_train = download_data("eduhuemar001/dataset-sentiment-4bit-train-v7", "train_dataset.json")

### Testing functions

In [ ]:
def evaluate_model(model, tokenizer, df):
    instruction = (
        "### Instruction:\n"
        "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
        "### Bewertung:\n"
    )
    answer_prefix = "\n\n### Antwort:\n"
    valid_outputs = ["positive", "neutral", "negative"]

    print(f"Evaluating TinyLlama model on {device.upper()}")

    pred_labels = []
    true_labels = []
    valid_count = 0

    start_inf_wall = time.time()

    if device == "cpu":
        start_inf_cpu = time.process_time()
        process = psutil.Process(os.getpid())
        mem_before = process.memory_info().rss

    # OOM prevention setup
    tokenizer.truncation_side = "left"
    tokenizer.model_max_length = 2048
    model.config.use_cache = False
    model.generation_config.use_cache = False

    for i, row in df.iterrows():
        text = row["review_text"]
        true_label = row["sentiment"]

        prompt = instruction + text + answer_prefix
        #inputs = tokenizer(prompt, return_tensors="pt").to(device)
        #with torch.no_grad():
        #    output = model.generate(**inputs, max_new_tokens=2)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=2046, padding=False).to(device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=2,
                                    do_sample=False, use_cache=False,
                                    pad_token_id=tokenizer.eos_token_id)

        decoded = tokenizer.decode(output[0], skip_special_tokens=True)

        if "### Antwort:" in decoded:
            answer = decoded.split("### Antwort:")[-1].strip().lower().split()[0]
        else:
            answer = decoded.strip().lower().split()[0]

        pred_labels.append(answer)
        true_labels.append(true_label)

        if answer in valid_outputs:
            valid_count += 1

    end_inf_wall = time.time()
    print(f"Inference wallclock time: {end_inf_wall - start_inf_wall:.2f}s")

    if device == "cpu":
        end_inf_cpu = time.process_time()
        mem_after = process.memory_info().rss if device == "cpu" else None
        mem_used_mb = (mem_after - mem_before) / (1024 ** 2)
        print(f"Inference CPU time: {end_inf_cpu - start_inf_cpu:.2f}s")
        print(f"\nMemory usage increase during inference: {mem_used_mb:.2f} MB")

    total_count = len(pred_labels)
    valid_percentage = 100 * valid_count / total_count
    print(f"\nGültige Modellantworten: {valid_count} von {total_count} ({valid_percentage:.2f}%)")
    print("\nClassification report:")
    print(classification_report(true_labels, pred_labels, digits=3))

In [ ]:
def evaluate_model_conf_matrix(model, tokenizer, df):
    instruction = (
        "### Instruction:\n"
        "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
        "### Bewertung:\n"
    )
    answer_prefix = "\n\n### Antwort:\n"

    # fixed label order for metrics & confusion matrix
    labels = ["positive", "neutral", "negative"]
    label_set = set(labels)

    print(f"Evaluating TinyLlama model on {device.upper()}")

    pred_labels, true_labels = [], []
    valid_count = 0

    start_inf_wall = time.time()
    if device == "cpu":
        start_inf_cpu = time.process_time()
        process = psutil.Process(os.getpid())
        mem_before = process.memory_info().rss

    # OOM prevention setup
    tokenizer.truncation_side = "left"
    tokenizer.model_max_length = 2048
    model.config.use_cache = False
    model.generation_config.use_cache = False

    for _, row in df.iterrows():
        text = str(row["review_text"])
        true_label = str(row["sentiment"]).lower().strip()

        prompt = instruction + text + answer_prefix
        #inputs = tokenizer(prompt, return_tensors="pt").to(device)
        #with torch.no_grad():
        #    output = model.generate(**inputs, max_new_tokens=2)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=2046, padding=False).to(device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=2,
                                    do_sample=False, use_cache=False,
                                    pad_token_id=tokenizer.eos_token_id)

        decoded = tokenizer.decode(output[0], skip_special_tokens=True)
        if "### Antwort:" in decoded:
            answer = decoded.split("### Antwort:")[-1].strip().lower()
        else:
            answer = decoded.strip().lower()

        # take first token/word as the label prediction
        pred = (answer.split() + [""])[0]

        # track validity for reporting
        if pred in label_set:
            valid_count += 1
        else:
            # clamp invalid predictions to a neutral fallback for metrics
            pred = "neutral"

        pred_labels.append(pred)
        true_labels.append(true_label)

    end_inf_wall = time.time()
    print(f"Inference wallclock time: {end_inf_wall - start_inf_wall:.2f}s")

    if device == "cpu":
        end_inf_cpu = time.process_time()
        mem_after = process.memory_info().rss
        mem_used_mb = (mem_after - mem_before) / (1024 ** 2)
        print(f"Inference CPU time: {end_inf_cpu - start_inf_cpu:.2f}s")
        print(f"Memory usage increase during inference: {mem_used_mb:.2f} MB")

    total_count = len(pred_labels)
    valid_pct = 100 * valid_count / total_count if total_count else 0.0
    print(f"\nGültige Modellantworten: {valid_count} von {total_count} ({valid_pct:.2f}%)")

    # classification report
    print("\nClassification report:")
    print(classification_report(true_labels, pred_labels, digits=3, zero_division=0))

    # confusion matrix (3x3, ordered by labels)
    cm = confusion_matrix(true_labels, pred_labels, labels=labels)
    cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
    print("\nConfusion matrix:")
    print(cm_df)

    # per-class TP, FP, TN, FN
    tp_fp_tn_fn_rows = []
    N = cm.sum()
    row_sums = cm.sum(axis=1)
    col_sums = cm.sum(axis=0)

    for i, lab in enumerate(labels):
        TP = cm[i, i]
        FP = col_sums[i] - TP
        FN = row_sums[i] - TP
        TN = N - TP - FP - FN
        tp_fp_tn_fn_rows.append(
            {"label": lab, "TP": int(TP), "FP": int(FP), "TN": int(TN), "FN": int(FN)}
        )

    tptn_df = pd.DataFrame(tp_fp_tn_fn_rows).set_index("label")
    print("\n")
    print(tptn_df)
    print("\n")
    return {
        "report": classification_report(true_labels, pred_labels, labels=labels, digits=3, zero_division=0, output_dict=True),
        "confusion_matrix": cm_df,
        "per_class_counts": tptn_df,
        "valid_percentage": valid_pct
    }

In [ ]:
def evaluate_bert_model(model, tokenizer, df):
    print(f"Evaluating BERT model on {device.upper()}")

    start_inf_wall = time.time()

    if device == "cpu":
        start_inf_cpu = time.process_time()
        process = psutil.Process(os.getpid())
        mem_before = process.memory_info().rss
        clf_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=-1)
    else:
        clf_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)

    # Run inference
    pred_labels = []
    true_labels = []
    valid_outputs = {"positive", "neutral", "negative"}
    valid_count = 0

    texts = df["review_text"].tolist()
    true_labels = df["sentiment"].tolist()

    results = clf_pipeline(texts, truncation=True)

    for res in results:
        label = res["label"].lower()
        pred_labels.append(label)
        if label in valid_outputs:
            valid_count += 1

    end_inf_wall = time.time()
    print(f"Inference wallclock time: {end_inf_wall - start_inf_wall:.2f}s")

    if device == "cpu":
        end_inf_cpu = time.process_time()
        mem_after = process.memory_info().rss
        mem_used_mb = (mem_after - mem_before) / (1024 ** 2)
        print(f"Inference CPU time: {end_inf_cpu - start_inf_cpu:.2f}s")
        print(f"\nMemory usage increase during inference: {mem_used_mb:.2f} MB")

    total = len(df)
    valid_percentage = 100 * valid_count / total
    print(f"\nGültige Modellantworten: {valid_count} von {total} ({valid_percentage:.2f}%)")
    print("\nClassification report:")
    print(classification_report(true_labels, pred_labels, digits=3))

### Execute when CPU active

In [ ]:
evaluate_model(model, tokenizer, df_balanced)

Evaluating TinyLlama model on CPU


Token indices sequence length is longer than the specified maximum sequence length for this model (5039 > 2048). Running this sequence through the model will result in indexing errors
This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


Inference wallclock time: 7379.74s
Inference CPU time: 42004.17s

Memory usage increase during inference: 3433.34 MB

Gültige Modellantworten: 1529 von 3000 (50.97%)

Classification report:
              precision    recall  f1-score   support

         "er      0.000     0.000     0.000         0
          "f      0.000     0.000     0.000         0
          "i      0.000     0.000     0.000         0
        "pos      0.000     0.000     0.000         0
           +      0.000     0.000     0.000         0
         +++      0.000     0.000     0.000         0
        +and      0.000     0.000     0.000         0
          +e      0.000     0.000     0.000         0
          +m      0.000     0.000     0.000         0
           -      0.000     0.000     0.000         0
        @ber      0.000     0.000     0.000         0
         @db      0.000     0.000     0.000         0
          @k      0.000     0.000     0.000         0
         @lo      0.000     0.000     0.000         0

C:\Users\marku\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marku\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marku\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marku\anaconda3\Lib\site-packages\sklea

In [ ]:
evaluate_model(lora_model, tokenizer, df_balanced)

Evaluating TinyLlama model on CPU


Token indices sequence length is longer than the specified maximum sequence length for this model (2670 > 2048). Running this sequence through the model will result in indexing errors
This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


Inference wallclock time: 11394.62s
Inference CPU time: 11206.61s

Memory usage increase during inference: 7.14 MB

Gültige Modellantworten: 147 von 150 (98.00%)

Classification report:
              precision    recall  f1-score   support

    negative      0.519     0.840     0.641        50
     neutral      0.850     0.340     0.486        50
    positive      0.739     0.680     0.708        50
         sch      0.000     0.000     0.000         0
       thank      0.000     0.000     0.000         0

    accuracy                          0.620       150
   macro avg      0.422     0.372     0.367       150
weighted avg      0.703     0.620     0.612       150



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
evaluate_bert_model(bert_model, bert_tokenizer, df_balanced)

Device set to use cpu


Evaluating BERT model on CPU
Inference wallclock time: 276.60s
Inference CPU time: 1640.45s

Memory usage increase during inference: 458.81 MB

Gültige Modellantworten: 3000 von 3000 (100.00%)

Classification report:
              precision    recall  f1-score   support

    negative      0.659     0.751     0.702      1000
     neutral      0.784     0.677     0.726      1000
    positive      0.835     0.832     0.833      1000

    accuracy                          0.753      3000
   macro avg      0.759     0.753     0.754      3000
weighted avg      0.759     0.753     0.754      3000



### Execute when GPU active

In [ ]:
evaluate_model_conf_matrix(model, tokenizer, df_balanced_test)

Evaluating TinyLlama model on CUDA
Inference wallclock time: 24.10s

Gültige Modellantworten: 83 von 150 (55.33%)

Classification report:
              precision    recall  f1-score   support

    positive      0.325     0.540     0.406        50
     neutral      0.254     0.340     0.291        50
    negative      0.000     0.000     0.000        50

    accuracy                          0.293       150
   macro avg      0.193     0.293     0.232       150
weighted avg      0.193     0.293     0.232       150


Confusion matrix:
               pred_positive  pred_neutral  pred_negative
true_positive             27            23              0
true_neutral              33            17              0
true_negative             23            27              0


          TP  FP   TN  FN
label                    
positive  27  56   44  23
neutral   17  50   50  33
negative   0   0  100  50




{'report': {'positive': {'precision': 0.3253012048192771,
   'recall': 0.54,
   'f1-score': 0.40601503759398494,
   'support': 50.0},
  'neutral': {'precision': 0.2537313432835821,
   'recall': 0.34,
   'f1-score': 0.2905982905982906,
   'support': 50.0},
  'negative': {'precision': 0.0,
   'recall': 0.0,
   'f1-score': 0.0,
   'support': 50.0},
  'accuracy': 0.29333333333333333,
  'macro avg': {'precision': 0.19301084936761972,
   'recall': 0.2933333333333334,
   'f1-score': 0.2322044427307585,
   'support': 150.0},
  'weighted avg': {'precision': 0.19301084936761972,
   'recall': 0.29333333333333333,
   'f1-score': 0.23220444273075855,
   'support': 150.0}},
 'confusion_matrix':                pred_positive  pred_neutral  pred_negative
 true_positive             27            23              0
 true_neutral              33            17              0
 true_negative             23            27              0,
 'per_class_counts':           TP  FP   TN  FN
 label                    


In [ ]:
evaluate_model_conf_matrix(lora_model, tokenizer, df_balanced_test)
#evaluate_model_conf_matrix(lora_model, tokenizer, df_test)

Evaluating TinyLlama model on CUDA
Inference wallclock time: 710.13s

Gültige Modellantworten: 3000 von 3000 (100.00%)

Classification report:
              precision    recall  f1-score   support

    negative      0.815     0.820     0.818      1000
     neutral      0.888     0.873     0.880      1000
    positive      0.878     0.888     0.883      1000

    accuracy                          0.860      3000
   macro avg      0.861     0.860     0.860      3000
weighted avg      0.861     0.860     0.860      3000


Confusion matrix:
               pred_positive  pred_neutral  pred_negative
true_positive            888            27             85
true_neutral              26           873            101
true_negative             97            83            820


           TP   FP    TN   FN
label                        
positive  888  123  1877  112
neutral   873  110  1890  127
negative  820  186  1814  180




{'report': {'positive': {'precision': 0.8783382789317508,
   'recall': 0.888,
   'f1-score': 0.8831427150671308,
   'support': 1000.0},
  'neutral': {'precision': 0.8880976602238047,
   'recall': 0.873,
   'f1-score': 0.8804841149773072,
   'support': 1000.0},
  'negative': {'precision': 0.8151093439363817,
   'recall': 0.82,
   'f1-score': 0.8175473579262214,
   'support': 1000.0},
  'accuracy': 0.8603333333333333,
  'macro avg': {'precision': 0.8605150943639791,
   'recall': 0.8603333333333333,
   'f1-score': 0.8603913959902197,
   'support': 3000.0},
  'weighted avg': {'precision': 0.860515094363979,
   'recall': 0.8603333333333333,
   'f1-score': 0.8603913959902199,
   'support': 3000.0}},
 'confusion_matrix':                pred_positive  pred_neutral  pred_negative
 true_positive            888            27             85
 true_neutral              26           873            101
 true_negative             97            83            820,
 'per_class_counts':            TP   FP 

In [ ]:
evaluate_model_conf_matrix(lora_model, tokenizer, df_balanced_train)
#evaluate_model_conf_matrix(lora_model, tokenizer, df_train)

Evaluating TinyLlama model on CUDA
Inference wallclock time: 687.88s

Gültige Modellantworten: 3000 von 3000 (100.00%)

Classification report:
              precision    recall  f1-score   support

    negative      0.975     0.969     0.972      1000
     neutral      0.988     0.987     0.987      1000
    positive      0.974     0.981     0.978      1000

    accuracy                          0.979      3000
   macro avg      0.979     0.979     0.979      3000
weighted avg      0.979     0.979     0.979      3000


Confusion matrix:
               pred_positive  pred_neutral  pred_negative
true_positive            981             4             15
true_neutral               3           987             10
true_negative             23             8            969


           TP  FP    TN  FN
label                      
positive  981  26  1974  19
neutral   987  12  1988  13
negative  969  25  1975  31




{'report': {'positive': {'precision': 0.974180734856008,
   'recall': 0.981,
   'f1-score': 0.9775784753363229,
   'support': 1000.0},
  'neutral': {'precision': 0.987987987987988,
   'recall': 0.987,
   'f1-score': 0.9874937468734367,
   'support': 1000.0},
  'negative': {'precision': 0.9748490945674044,
   'recall': 0.969,
   'f1-score': 0.9719157472417251,
   'support': 1000.0},
  'accuracy': 0.979,
  'macro avg': {'precision': 0.9790059391371334,
   'recall': 0.979,
   'f1-score': 0.9789959898171615,
   'support': 3000.0},
  'weighted avg': {'precision': 0.9790059391371334,
   'recall': 0.979,
   'f1-score': 0.9789959898171616,
   'support': 3000.0}},
 'confusion_matrix':                pred_positive  pred_neutral  pred_negative
 true_positive            981             4             15
 true_neutral               3           987             10
 true_negative             23             8            969,
 'per_class_counts':            TP  FP    TN  FN
 label                      
 

In [ ]:
evaluate_bert_model(bert_model, bert_tokenizer, df_balanced)

Device set to use cuda:0


Evaluating BERT model on CUDA
Inference wallclock time: 35.99s

Gültige Modellantworten: 3000 von 3000 (100.00%)

Classification report:
              precision    recall  f1-score   support

    negative      0.659     0.751     0.702      1000
     neutral      0.784     0.677     0.726      1000
    positive      0.835     0.832     0.833      1000

    accuracy                          0.753      3000
   macro avg      0.759     0.753     0.754      3000
weighted avg      0.759     0.753     0.754      3000



### Lora model testing (old)

In [ ]:
true_labels_lora = []
pred_labels_lora = []
valid_outputs = ["positive", "neutral", "negative"]
valid_count = 0

for i, row in df_balanced.iterrows():
    text = row["review_text"]
    true_label = row["sentiment"]

    prompt = instruction + text + answer_prefix
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # output = model.generate(**inputs, max_new_tokens=2)
    output = lora_model.generate(**inputs, max_new_tokens=2)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract raw model answer
    if "### Antwort:" in decoded:
        answer = decoded.split("### Antwort:")[-1].strip().lower()
        answer = answer.split()[0] if answer.split() else ""
    else:
        answer = decoded.strip().lower()

    # Track output
    pred_labels_lora.append(answer)
    true_labels_lora.append(true_label)

    if answer in valid_outputs:
        valid_count += 1

    print(f"\n[{i+1}] Bewertung: {text}")
    print(f"    Wahre Stimmung: {true_label}")
    print(f"    Modellantwort: {answer}")

# Validity stats
total_count = len(pred_labels_lora)
valid_percentage = 100 * valid_count / total_count
print(f"\nGültige Modellantworten: {valid_count} von {total_count} ({valid_percentage:.2f}%)")

# Classification report

print("\nKlassifikationsbericht:")
print(classification_report(true_labels_lora, pred_labels_lora, digits=3))

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
    Modellantwort: positive

[1177] Bewertung: »Ein Gefährdungspotenzial was wirklich nicht nötig täte« Bahn-Unterführung passiert und sich im finsteren Wald wieder findet. Das sei doch gefährlich. Viele Bürger haben sich inzwischen bei TA beschwert. Doch die Lampen bleiben wohl aus, die Laternenköpfe wurden sogar demontiert
    Wahre Stimmung: neutral
    Modellantwort: negative

[1701] Bewertung: Re: Stralsund - HST "kann doch nur ein ""Feuerteufel"" gewesen sein. So oft wie das in letzter Zeit dort gebrannt hat.Meines Wissens nach hat die Bahn das haus im letzten Jahr verkauft. Wenn jetzt auch noch der letzte Mieter raus ist........."
    Wahre Stimmung: neutral
    Modellantwort: neutral

[1041] Bewertung: Nach Ansturm auf Wank: Das will die BZB ändern | Garmisch-Partenkirchen gekommen – zu Fuß oder per Bahn. „Wir sind ja begeistert, dass diese Veranstaltung so gut angenommen wurde“, betont BZB-Marketingleiter Klaus

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Base model testing (old)

In [ ]:
true_labels_base = []
pred_labels_base = []
valid_outputs = ["positive", "neutral", "negative"]
valid_count = 0

for i, row in df_balanced.iterrows():
    text = row["review_text"]
    true_label = row["sentiment"]

    prompt = instruction + text + answer_prefix
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(**inputs, max_new_tokens=2)
    # output = lora_model.generate(**inputs, max_new_tokens=2)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract raw model answer
    if "### Antwort:" in decoded:
        answer = decoded.split("### Antwort:")[-1].strip().lower()
        answer = answer.split()[0] if answer.split() else ""
    else:
        answer = decoded.strip().lower()

    # Track output
    pred_labels_base.append(answer)
    true_labels_base.append(true_label)

    if answer in valid_outputs:
        valid_count += 1

    print(f"\n[{i+1}] Bewertung: {text}")
    print(f"    Wahre Stimmung: {true_label}")
    print(f"    Modellantwort: {answer}")

# Validity stats
total_count = len(pred_labels_base)
valid_percentage = 100 * valid_count / total_count
print(f"\nGültige Modellantworten: {valid_count} von {total_count} ({valid_percentage:.2f}%)")

# Classification report

print("\nKlassifikationsbericht (nur gültige Vorhersagen):")
print(classification_report(true_labels_base, pred_labels_base, digits=3))

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
[1302] Bewertung: Re: Deutsche Bahn Karriere Hallo Deutsche Bahn Karriere, wie kann ich Kontakt mit der DB-Sicherheit in Frankfurt am Main aufnehmen ?
    Wahre Stimmung: neutral
    Modellantwort: ich

[2366] Bewertung: Eine sehr grosse Enttaeuschung war der Fitness-Club. - Der Crosstrainer ist notduerftig repariert und in diesem Zustand lebensgefaehrlich. -Die zwei Laufbaender waren okay, allerdings auch mit kleineren Defekten, da die Notfall-Vorrichtung staendig aus dem Geraet rausgeht und somit das Band stehen bleibt. -Ferner steht ein defekter Bauchtrainer im Raum, bei dem der Sitz nur mit 2 Schrauben befestigt ist und man sich auch sehr schlimm verletzen kann. Ein Hotel, dass "Fitness" im Namen fuehrt, sollte auf jeden Fall etwas mehr als diese wenigen, defekten Geraete bieten.Die SPA-Anlage ist masslos ueberteuert. So verlangt man fuer deinen Kinder-Haarschnitt 20 Euro. Das sind Preise, die teurer sind als in Deu

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_